> **LEGACY single-hand test.** This notebook does not mirror left hands, so left-hand signing will
> be less reliable here. Use `05_multi_hand_demo.ipynb` (`src/recognizer.py`), which does.

# Task 2: Live Recognition

This is the real deliverable of Task 2: continuous, real-time recognition of
dynamic signs from the webcam, using the classifier trained in
`03_train_classifier.ipynb`.

Pipeline per frame:
1. Detect the hand, extract normalized landmarks.
2. Keep a rolling buffer of the last ~1.2s of landmarks.
3. **Motion gate**: only attempt classification when the hand is actually
   moving enough — otherwise a resting/idle hand would constantly get fed
   into the classifier as if it were mid-gesture.
4. Resample the buffer to the same fixed length used in training, classify it.
5. **Confirm** a detection only once confidence clears a threshold, and
   **cooldown** afterwards so one gesture isn't counted twice.
6. Emit a clean event — this is what Task 3 (identity) and Task 4 (game
   control) would consume.

Run cells top to bottom. Kernel: **Python (ipcv-task2)**.

## 1. Config

In [ ]:
BUFFER_SECONDS = 1.2        # should match CLIP_DURATION used in data collection
MOTION_THRESHOLD = 0.03     # min motion energy to attempt classification — tune in step 3 below
MIN_SPAN_FRACTION = 0.5     # classify once the buffer holds this fraction of BUFFER_SECONDS of motion.
                            # (0.7 meant waiting 0.84 s — most ASL words are ~1 s, so short signs were
                            #  over before the system would guess; 0.5 confirmed 14/24 held-out signs vs 11/24)

# Confirmation rule: a sign is confirmed when the SAME sign stays the top
# prediction, with probability >= CONFIDENCE_THRESHOLD, for CONFIRM_SECONDS.
# (A 25-class Random Forest spreads its votes thin: correct predictions have a
#  median top probability of only ~0.5, so a 0.6 cut-off confirmed just ~26% of
#  clips. 0.35 confirms ~60% at ~90% precision; the hold time protects against
#  the extra false confirmations a lower threshold would otherwise allow.)
CONFIDENCE_THRESHOLD = 0.30
CONFIRM_SECONDS = 0.2       # fixed, equal delay for every player — report it
COOLDOWN_SECONDS = 1.0      # ignore new confirmations for this long after one fires
IDLE_LABEL = "Idle"         # must match the idle class name used in data collection

## 2. Imports and model loading

In [ ]:
import sys, os, importlib
sys.path.append(os.path.abspath(".."))

import time
from collections import deque

import cv2
import mediapipe as mp
import numpy as np
import joblib

import src.features, src.hand_model
# Pick up edits to src/ without restarting the kernel (a kernel keeps the
# first-imported copy of a module in memory otherwise).
importlib.reload(src.features)
importlib.reload(src.hand_model)

from src.features import extract_frame, build_clip_feature, motion_energy
from src.hand_model import load_hand_landmarker

landmarker = load_hand_landmarker(model_path="../models/hand_landmarker.task", num_hands=1)

bundle = joblib.load("../models/sign_classifier.joblib")
clf = bundle["model"]
TARGET_FRAMES = bundle["target_frames"]
print(f"Loaded classifier. Classes: {bundle['classes']}")
print(f"Target frames per window: {TARGET_FRAMES}")

## 3. Calibrate the motion threshold (optional but recommended)

Run this for ~10 seconds: first hold your hand still, then perform a sign.
Watch the printed motion-energy values to pick a `MOTION_THRESHOLD` in step 1
that clearly separates "resting" from "gesturing".

In [ ]:
calib_buffer = deque()
cap = cv2.VideoCapture(0)
start = time.time()

try:
    while time.time() - start < 10:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)

        if result.hand_landmarks:
            raw63, _ = extract_frame(result.hand_landmarks[0], aspect=frame.shape[1] / frame.shape[0])
            calib_buffer.append(raw63)
            if len(calib_buffer) > 10:
                calib_buffer.popleft()
            if len(calib_buffer) >= 2:
                energy = motion_energy(list(calib_buffer))
                print(f"motion energy: {energy:.4f}", end="\r")

        cv2.imshow("Calibration - hold still, then gesture (10s)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.waitKey(1)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

print("\nDone. Pick MOTION_THRESHOLD in step 1 to sit between your 'still' and 'gesturing' values.")

## 4. Live recognition loop

Press **q** to quit. Confirmed events are printed to the console with a
timestamp, the recognized sign, and confidence — this is the exact kind of
output Task 3/Task 4 would consume.

In [ ]:
FONT = cv2.FONT_HERSHEY_SIMPLEX

buffer = deque()  # list of (timestamp, raw63, hand_scale)
last_confirmed_time = 0.0
last_confirmed_label = None
flash_until = 0.0  # show the confirmed label on screen briefly
streak_label, streak_start = None, 0.0  # sign currently being held, and since when

cap = cv2.VideoCapture(0)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame grab failed, stopping.")
            break

        now = time.time()
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)
        hand_visible = len(result.hand_landmarks) > 0

        if hand_visible:
            raw63, hand_scale = extract_frame(result.hand_landmarks[0], aspect=frame.shape[1] / frame.shape[0])
            buffer.append((now, raw63, hand_scale))

        # drop anything older than BUFFER_SECONDS
        while buffer and now - buffer[0][0] > BUFFER_SECONDS:
            buffer.popleft()

        predicted_label, confidence = None, 0.0
        energy = 0.0

        if len(buffer) >= 5:
            times = [t for t, _, _ in buffer]
            feats = [f for _, f, _ in buffer]
            scales = [s for _, _, s in buffer]
            energy = motion_energy(feats)

            # Only classify if there's enough motion AND the buffer spans
            # close to the full window (so we're not classifying a half-formed gesture)
            span = times[-1] - times[0]
            if energy >= MOTION_THRESHOLD and span >= BUFFER_SECONDS * MIN_SPAN_FRACTION:
                seq = build_clip_feature(feats, scales, times, TARGET_FRAMES)
                if seq is not None:
                    probs = clf.predict_proba([seq])[0]
                    best_idx = np.argmax(probs)
                    predicted_label = clf.classes_[best_idx]
                    confidence = probs[best_idx]

        # --- stability rule: same sign, above threshold, held for CONFIRM_SECONDS ---
        held = 0.0
        if predicted_label is not None and predicted_label != IDLE_LABEL and confidence >= CONFIDENCE_THRESHOLD:
            if predicted_label == streak_label:
                held = now - streak_start
            else:
                streak_label, streak_start = predicted_label, now
            if held >= CONFIRM_SECONDS and (now - last_confirmed_time) >= COOLDOWN_SECONDS:
                last_confirmed_time = now
                last_confirmed_label = predicted_label
                flash_until = now + 1.0
                streak_label = None
                buffer.clear()  # avoid re-triggering on the tail of this gesture
                print(f"[{time.strftime('%H:%M:%S')}] CONFIRMED: {predicted_label}  (confidence {confidence:.2f})")
        else:
            streak_label = None

        # --- overlay ---
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 110), (40, 40, 40), -1)
        if not hand_visible:
            cv2.putText(frame, "No hand detected", (20, 40), FONT, 0.8, (0, 0, 255), 2)
        else:
            cv2.putText(frame, f"motion: {energy:.3f}  (gate: {MOTION_THRESHOLD})", (20, 30), FONT, 0.6, (200, 200, 200), 1)
            if predicted_label is not None:
                hold_txt = f"  holding {min(held, CONFIRM_SECONDS):.1f}/{CONFIRM_SECONDS}s" if streak_label else ""
                cv2.putText(frame, f"guess: {predicted_label} ({confidence:.2f}){hold_txt}", (20, 60), FONT, 0.7, (0, 255, 255), 2)

        if now < flash_until:
            cv2.putText(frame, f"CONFIRMED: {last_confirmed_label}", (20, 95), FONT, 0.9, (0, 255, 0), 3)

        cv2.imshow("Task 2 - Live Dynamic Sign Recognition (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.waitKey(1)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

## Cleanup

In [ ]:
landmarker.close()
print("Landmarker closed.")

## What to evaluate from here (for your report)

- **Accuracy**: already measured in `03_train_classifier.ipynb` (confusion matrix).
- **Latency**: time from when you finish a gesture to when "CONFIRMED" prints — roughly `BUFFER_SECONDS` plus `CONFIRM_SECONDS` plus classification time, since the buffer must fill and the sign must be held before it confirms. Measure this concretely and report the number.
- **False positives**: does `Idle` ever get confused with a real sign, or does idle hand movement ever falsely confirm a sign? Try deliberately moving your hand randomly for a while and count any false triggers.
- **Failure cases**: try fast vs. slow gestures, partial occlusion, hand leaving frame mid-gesture — document what breaks and how it degrades (does it just silently fail to confirm, or does it misclassify?).
- **Fairness note for the game**: since this is a race-to-be-first game, the roughly `BUFFER_SECONDS` + cooldown delay is a *fixed, equal* delay for every player — worth stating explicitly in the report.